# Bronze to silver

Raw sales go into `bronze_sales` exactly as collected. This notebook cleans them into `silver_sales`. Runs on Databricks (Free Edition, serverless).

In [ ]:
from pyspark.sql import functions as F

bronze = spark.read.table("workspace.default.bronze_sales")
print(bronze.count())

128614


## Silver

Proper types, wear and StatTrak taken out of the item name, a data quality flag on every row instead of dropping anything, and duplicates removed.

In [ ]:
silver = (
    bronze
    .withColumn("sale_ts", F.to_timestamp("ts"))
    .withColumn("sale_date", F.to_date("sale_ts"))
    .withColumn("price", F.col("price").cast("double"))
    .withColumn("markup_pct", F.col("markup_pct").cast("double"))
    .withColumn("buff_price", F.col("buff_price").cast("double"))
    .withColumn("csfloat_price", F.col("csfloat_price").cast("double"))
    .withColumn("stattrak", F.col("item").contains("StatTrak"))
    .withColumn("wear", F.regexp_extract("item", r"\((Factory New|Minimal Wear|Field-Tested|Well-Worn|Battle-Scarred)\)$", 1))
    .withColumn("dq_missing_buff", F.col("buff_price").isNull())
    .withColumn("dq_missing_csfloat", F.col("csfloat_price").isNull())
    .withColumn("dq_extreme_markup", (F.col("markup_pct") > 100) | (F.col("markup_pct") < -50))
    .dropDuplicates(["ts", "item", "price"])
    .drop("ts")
)
print(silver.count())

128613


Only one duplicate out of 128,614 rows, so the collection itself is clean.

In [ ]:
silver.write.mode("overwrite").format("delta").saveAsTable("workspace.default.silver_sales")

display(spark.sql("""
SELECT sale_date, COUNT(*) AS sales, ROUND(PERCENTILE(price, 0.5), 2) AS median_price,
       SUM(CAST(dq_extreme_markup AS INT)) AS extreme_markups
FROM workspace.default.silver_sales
GROUP BY sale_date ORDER BY sale_date DESC LIMIT 10
"""))

sale_date,sales,median_price,extreme_markups
2026-09-25,1188,181.75,4
2026-09-24,1018,150.8,2
2026-09-23,912,142.17,2
2026-09-22,953,160.82,3
2026-09-21,832,162.98,3
2026-09-20,944,138.11,3
2026-09-19,942,146.27,5
2026-09-18,1174,157.77,2
2026-09-17,989,174.11,2
2026-09-16,981,163.91,0
